In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

# Data Reading

In [0]:
df = spark.read.format("parquet")\
    .option("header", True)\
    .option("inferSchema", True)\
    .load("abfss://bronze@databrikcsete.dfs.core.windows.net/products")

In [0]:
display(df)

In [0]:
df = df.drop("_rescued_data")

display(df)

In [0]:
df.createOrReplaceTempView("products")


### Functions

In [0]:
%sql
CREATE OR REPLACE FUNCTION databricks_cata.bronze.discount_func(p_price DOUBLE)
RETURNS DOUBLE
LANGUAGE SQL
RETURN p_price * 0.90

In [0]:
%sql
select  *,
       databricks_cata.bronze.discount_func(price) as discounted_price
       from products
       

In [0]:
df = df.withColumn("discounted_price", expr("databricks_cata.bronze.discount_func(price)"))

display(df)

In [0]:
df.write.format("delta")\
  .mode("append")\
  .option("path", "abfss://silver@databrikcsete.dfs.core.windows.net/products")\
  .save()  